# 01. data quality

## Goal
The goal of this notebook is to check the quality and consistency of the data before deeper EDA and modeling.

## Why this matters
The dataset may contain missing values, duplicate rows, merge issues, MCC conflicts, or other anomalies. We need to find these issues, document them, and decide how they should be handled before feature engineering and modeling.

In [3]:
import pandas as pd
from mdq.data import load_data

In [2]:
business, consumer, merchant, mcc = load_data()

In [4]:
print(business.duplicated().sum())
print(consumer.duplicated().sum())
print(merchant.duplicated().sum())

0
0
0


In [5]:
print(consumer.isnull().sum())

transaction_date          0
transaction_timestamp     0
transaction_amount_kzt    0
mcc                       0
merchant_id               0
channel                   0
bank_name                 0
country                   0
card_number               0
card_tier                 0
tokenized                 0
is_recurring              0
dtype: int64


In [6]:
print(business.isnull().sum())

transaction_date          0
transaction_timestamp     0
transaction_amount_kzt    0
mcc                       0
merchant_id               0
channel                   0
bank_name                 0
country                   0
card_number               0
card_tier                 0
tokenized                 0
is_recurring              0
dtype: int64


In [7]:
print(merchant.isnull().sum())

merchant_id          0
merchant_name        0
mcc                  0
merchant_country     0
recurring_capable    0
dtype: int64


In [8]:
merchant["merchant_id"].duplicated().sum()

np.int64(0)

## There are no duplicate rows or missing values

In [9]:
business_merge_check = business.merge(
    merchant,
    on="merchant_id",
    how="left",
    indicator=True
)

consumer_merge_check = consumer.merge(
    merchant,
    on="merchant_id",
    how="left",
    indicator=True
)

print("business merge:")
print(business_merge_check["_merge"].value_counts(normalize=True))

print("consumer merge:")
print(consumer_merge_check["_merge"].value_counts(normalize=True))

business merge:
_merge
both          1.0
left_only     0.0
right_only    0.0
Name: proportion, dtype: float64
consumer merge:
_merge
both          1.0
left_only     0.0
right_only    0.0
Name: proportion, dtype: float64


In [11]:
print("business before:", len(business))
print("business after merge:", len(business_merge_check))

print("consumer before:", len(consumer))
print("consumer after merge:", len(consumer_merge_check))

business before: 2997593
business after merge: 2997593
consumer before: 9832487
consumer after merge: 9832487


## No issues after merging

In [12]:
for name, df in [("business", business), ("consumer", consumer)]:
    parsed = pd.to_datetime(df["transaction_timestamp"], errors="coerce")
    print(name, "invalid timestamps:", parsed.isna().sum())

business invalid timestamps: 0
consumer invalid timestamps: 0


In [13]:
for name, df in [("business", business), ("consumer", consumer)]:
    print(name)
    print("negative:", (df["transaction_amount_kzt"] < 0).sum())
    print("zero:", (df["transaction_amount_kzt"] == 0).sum())
    print("positive:", (df["transaction_amount_kzt"] > 0).sum())

business
negative: 0
zero: 0
positive: 2997593
consumer
negative: 0
zero: 0
positive: 9832487


## No Issues Found in Basic Logic and Timestamp Checks

In [15]:
pd.options.display.float_format = "{:,.0f}".format

for name, df in [("business", business), ("consumer", consumer)]:
    print(name)
    display(df["transaction_amount_kzt"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.999]))

business


count    2,997,593
mean       156,535
std        252,868
min             67
50%         77,224
75%        196,081
90%        381,919
95%        566,273
99%      1,090,845
99.9%    2,546,925
max     40,799,297
Name: transaction_amount_kzt, dtype: float64

consumer


count    9,832,487
mean        54,045
std        169,655
min             15
50%         11,892
75%         39,665
90%        117,999
95%        222,106
99%        699,869
99.9%    2,151,535
max     31,971,032
Name: transaction_amount_kzt, dtype: float64

In [63]:
overlap_cards = set(business["card_number"]) & set(consumer["card_number"])

print("overlap cards:", len(overlap_cards))

overlap cards: 0


## No overlap was found between business and consumer card numbers.

In [65]:
merchant_mcc = merchant[["merchant_id", "mcc"]].rename(columns={"mcc": "merchant_mcc"})

for name, df in [("business", business), ("consumer", consumer)]:
    check = df.merge(merchant_mcc, on="merchant_id", how="left", validate="many_to_one")
    check["txn_mcc"] = check["mcc"].astype(str)
    check["merchant_mcc"] = check["merchant_mcc"].astype(str)

    conflict = check["txn_mcc"] != check["merchant_mcc"]

    print(name)
    print("mcc conflict rows:", conflict.sum())
    print("mcc conflict share:", conflict.mean())

    display(
        check.loc[conflict, ["merchant_id", "txn_mcc", "merchant_mcc"]]
        .value_counts()
        .head(20)
        .reset_index(name="rows")
    )

business
mcc conflict rows: 17701
mcc conflict share: 0.0059050711687677416


,merchant_id,txn_mcc,merchant_mcc,rows
0,MER_000000,7012,7311,17701


consumer
mcc conflict rows: 43150
mcc conflict share: 0.0043885133028907135


,merchant_id,txn_mcc,merchant_mcc,rows
0,MER_000000,7012,7311,43150


## Decision

We will use the MCC from the transaction table as the final MCC for EDA and modeling.

The reason is that transaction MCC describes the actual transaction category. Merchant MCC from the reference table is useful for validation, but it may not always match the transaction-level value.

The only detected conflict is for `MER_000000`, where some transactions have MCC `7012`, while the merchant reference has MCC `7311`. This issue is documented as a data quality anomaly.

Final decision:

- use transaction MCC as `mcc_final`;
- keep merchant reference MCC only for quality checks;
- mention the `MER_000000` conflict as a limitation.

In [68]:
for name, df in [("business", business), ("consumer", consumer)]:
    print(name)
    display(
        df.sort_values("transaction_amount_kzt", ascending=False)
        .head(10)
    )

business


,transaction_date,transaction_timestamp,transaction_amount_kzt,mcc,merchant_id,channel,bank_name,country,card_number,card_tier,tokenized,is_recurring
2695013,1767139200000000000,2025-12-31 11:12:50,40799297,7311,MER_000004,online,Halyk,Ireland,5438810053139120,Business,True,False
962900,1760400000000000000,2025-10-14 11:33:16,25866100,7311,MER_000005,online,Halyk,Kazakhstan,5100611454450984,Business,False,False
1251347,1766966400000000000,2025-12-29 06:09:06,21918401,7311,MER_000000,online,Halyk,Kazakhstan,5438815044966994,Business,False,False
859328,1762473600000000000,2025-11-07 11:07:03,20006303,7311,MER_000005,online,Kaspi,Kazakhstan,5228590263733573,Business,False,False
1409334,1771804800000000000,2026-02-23 09:02:49,19588407,7311,MER_000001,online,Forte Bank,Kazakhstan,5368293828851614,Business,True,False
146224,1762819200000000000,2025-11-11 17:49:34,19527459,5021,MER_000454,online,Kaspi,Kazakhstan,5531518204569121,Business,False,False
2271432,1774569600000000000,2026-03-27 16:10:59,19306307,7311,MER_000002,online,Kaspi,Kazakhstan,5531511670944001,Business,True,False
677351,1770163200000000000,2026-02-04 13:32:45,19094268,7311,MER_000005,online,Halyk,Ireland,5438814777783890,Business,True,False
457422,1772150400000000000,2026-02-27 13:33:19,18879538,7311,MER_000002,online,Kaspi,Kazakhstan,5531512482450351,Business,False,False
2713976,1769644800000000000,2026-01-29 14:21:11,14810546,7311,MER_000002,online,Kaspi,Kazakhstan,5176517937511863,Business,True,False


consumer


,transaction_date,transaction_timestamp,transaction_amount_kzt,mcc,merchant_id,channel,bank_name,country,card_number,card_tier,tokenized,is_recurring
7663138,1764892800000000000,2025-12-05 12:38:00,31971032,4511,MER_000067,POS,Kaspi,US,5531519163624030,Premium,False,False
4872950,1769990400000000000,2026-02-02 20:25:17,17457834,7311,MER_000005,online,Home Credit Bank,Kazakhstan,5201494837715409,Standard,False,False
373767,1759968000000000000,2025-10-09 17:32:00,14484034,8211,MER_000606,online,Halyk,Uzbekistan,5486737033379222,Affluent,True,False
5687298,1764115200000000000,2025-11-26 18:36:04,13908372,7311,MER_000000,POS,Kaspi,Kazakhstan,5531510798748252,Affluent,False,False
2318530,1772841600000000000,2026-03-07 13:29:23,13466034,7311,MER_000005,online,Freedom Bank,Kazakhstan,5498075374041275,Standard,False,False
1075433,1770854400000000000,2026-02-12 07:36:38,13089602,7311,MER_000002,online,Forte Bank,Singapore,5368299608783752,Affluent,False,False
2738003,1764115200000000000,2025-11-26 14:23:23,12404241,7311,MER_000005,online,Halyk,Ireland,5438810679490907,Affluent,False,False
2190159,1771977600000000000,2026-02-25 15:13:11,12286597,5099,MER_001012,online,Forte Bank,Kazakhstan,5168269516651435,Affluent,False,False
1035498,1765756800000000000,2025-12-15 16:41:26,12112386,7311,MER_000002,online,Forte Bank,Singapore,5368299608783752,Affluent,False,False
1002089,1761696000000000000,2025-10-29 07:23:25,12021528,7311,MER_000002,online,Forte Bank,Kazakhstan,5368299608783752,Affluent,False,False


## Top transaction amounts were inspected; no invalid negative/zero values were found